In [3]:
# VAE pour prédiction de Time Series S&P 500

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.losses import mse
import tensorflow as tf

# 1. Chargement des données
sp500 = yf.download('^GSPC', start='2010-01-01', end='2025-01-01')
data = sp500[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()

# 2. Normalisation
data_values = data.values
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data_values)

# 3. Construction des séquences
past_window = 30
future_window = 10
X, y = [], []
for i in range(len(data_scaled) - past_window - future_window):
    X.append(data_scaled[i:i+past_window])
    y.append(data_scaled[i+past_window:i+past_window+future_window])
X = np.array(X)
y = np.array(y)

# 4. Définition du VAE
latent_dim = 16
input_shape = X.shape[1:]

# Encodeur
encoder_inputs = Input(shape=input_shape)
x = layers.LSTM(64, return_sequences=False)(encoder_inputs)
z_mean = layers.Dense(latent_dim)(x)
z_log_var = layers.Dense(latent_dim)(x)

def sampling(args):
    z_mean, z_log_var = args
    epsilon = tf.random.normal(shape=(tf.shape(z_mean)[0], latent_dim))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = layers.Lambda(sampling)([z_mean, z_log_var])
encoder = Model(encoder_inputs, [z_mean, z_log_var, z], name='encoder')

# Décodeur
decoder_inputs = Input(shape=(latent_dim,))
x = layers.RepeatVector(future_window)(decoder_inputs)
x = layers.LSTM(64, return_sequences=True)(x)
decoder_outputs = layers.TimeDistributed(layers.Dense(input_shape[1]))(x)
decoder = Model(decoder_inputs, decoder_outputs, name='decoder')

# VAE Model
outputs = decoder(encoder(encoder_inputs)[2])
vae = Model(encoder_inputs, outputs, name='vae')

# Loss
reconstruction_loss = mse(y, outputs)
reconstruction_loss *= future_window * input_shape[1]
kl_loss = 1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
kl_loss = tf.reduce_mean(tf.reduce_sum(-0.5 * kl_loss, axis=1))
vae_loss = tf.reduce_mean(reconstruction_loss + kl_loss)
vae.add_loss(vae_loss)
vae.compile(optimizer='adam')

# 5. Entraînement
vae.fit(X, y, epochs=50, batch_size=64, validation_split=0.1)

# 6. Prédiction et visualisation
encoded_data = encoder.predict(X[:1])[2]
predicted_sequence = decoder.predict(encoded_data)

# Réaffichage dans l'échelle originale
true_future = scaler.inverse_transform(y[0])
pred_future = scaler.inverse_transform(predicted_sequence[0])

plt.figure(figsize=(10, 4))
plt.plot(true_future[:, 3], label='True Close Price')
plt.plot(pred_future[:, 3], label='Predicted Close Price', linestyle='--')
plt.title('S&P 500 Close Price Prediction')
plt.xlabel('Days Ahead')
plt.ylabel('Close Price')
plt.legend()
plt.grid(True)
plt.show()


[*********************100%***********************]  1 of 1 completed


Epoch 1/50


2025-04-04 13:29:35.780899: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - kl_loss: 0.0107 - loss: 0.0963 - reconstruction_loss: 0.0856

ValueError: No loss to compute. Provide a `loss` argument in `compile()`.